In [1]:
# import necessary libraries
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.metrics import r2_score
from scipy.stats import chi2_contingency
from statsmodels.stats.outliers_influence import variance_inflation_factor
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report, precision_recall_fscore_support
import warnings
import os

In [ ]:
#load datasets
a1 = pd.read_excel("E:\FraudPulse\Data\credit_risk\case_study1.xlsx")
a2 = pd.read_excel("E:\FraudPulse\Data\credit_risk\case_study2.xlsx")

In [ ]:
#making copies of the datasets to work with
df1 = a1.copy()
df2 = a2.copy()

In [6]:
#checking shapes 
print("Shape of df1:", df1.shape)
print("Shape of df2:", df2.shape)

Shape of df1: (51336, 26)
Shape of df2: (51336, 62)


In [7]:
#checking info of the datasets
print("\nInfo of df1:")
print(df1.info())
print("\nInfo of df2:")
print(df2.info())


Info of df1:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 51336 entries, 0 to 51335
Data columns (total 26 columns):
 #   Column                Non-Null Count  Dtype  
---  ------                --------------  -----  
 0   PROSPECTID            51336 non-null  int64  
 1   Total_TL              51336 non-null  int64  
 2   Tot_Closed_TL         51336 non-null  int64  
 3   Tot_Active_TL         51336 non-null  int64  
 4   Total_TL_opened_L6M   51336 non-null  int64  
 5   Tot_TL_closed_L6M     51336 non-null  int64  
 6   pct_tl_open_L6M       51336 non-null  float64
 7   pct_tl_closed_L6M     51336 non-null  float64
 8   pct_active_tl         51336 non-null  float64
 9   pct_closed_tl         51336 non-null  float64
 10  Total_TL_opened_L12M  51336 non-null  int64  
 11  Tot_TL_closed_L12M    51336 non-null  int64  
 12  pct_tl_open_L12M      51336 non-null  float64
 13  pct_tl_closed_L12M    51336 non-null  float64
 14  Tot_Missed_Pmnt       51336 non-null  int64  
 15  Auto_

In [8]:
#checking descriptive statistics
print("\nDescriptive statistics of df1:")
print(df1.describe())
print("\nDescriptive statistics of df2:")
print(df2.describe())


Descriptive statistics of df1:
         PROSPECTID      Total_TL  Tot_Closed_TL  Tot_Active_TL  \
count  51336.000000  51336.000000   51336.000000   51336.000000   
mean   25668.500000      4.858598       2.770415       2.088184   
std    14819.571046      7.177116       5.941680       2.290774   
min        1.000000      1.000000       0.000000       0.000000   
25%    12834.750000      1.000000       0.000000       1.000000   
50%    25668.500000      2.000000       1.000000       1.000000   
75%    38502.250000      5.000000       3.000000       3.000000   
max    51336.000000    235.000000     216.000000      47.000000   

       Total_TL_opened_L6M  Tot_TL_closed_L6M  pct_tl_open_L6M  \
count         51336.000000       51336.000000     51336.000000   
mean              0.736754           0.428919         0.184574   
std               1.296717           0.989972         0.297414   
min               0.000000           0.000000         0.000000   
25%               0.000000        

In [12]:
#checking head of datasets
print("\nHead of df1:") 
print(df1.head())
print("\nHead of df2:")
print(df2.head())



Head of df1:
   PROSPECTID  Total_TL  Tot_Closed_TL  Tot_Active_TL  Total_TL_opened_L6M  \
0           1         5              4              1                    0   
1           2         1              0              1                    0   
2           3         8              0              8                    1   
3           4         1              0              1                    1   
4           5         3              2              1                    0   

   Tot_TL_closed_L6M  pct_tl_open_L6M  pct_tl_closed_L6M  pct_active_tl  \
0                  0            0.000                0.0          0.200   
1                  0            0.000                0.0          1.000   
2                  0            0.125                0.0          1.000   
3                  0            1.000                0.0          1.000   
4                  0            0.000                0.0          0.333   

   pct_closed_tl  ...  CC_TL  Consumer_TL  Gold_TL  Home_TL  PL_TL

In [ ]:
#remove nulls
df1 = df1.loc[df1['Age_Oldest_TL']!=-99999]

In [14]:
column_to_be_removed = []
for i in df2.columns:
    if df2.loc[df2[i] == -99999].shape[0] > 10000:
        column_to_be_removed.append(i)

print("Columns to be removed due to high number of nulls:", column_to_be_removed)
        

Columns to be removed due to high number of nulls: ['time_since_first_deliquency', 'time_since_recent_deliquency', 'max_delinquency_level', 'max_deliq_6mts', 'max_deliq_12mts', 'CC_utilization', 'PL_utilization', 'max_unsec_exposure_inPct']


In [16]:
df2=df2.drop(columns=column_to_be_removed, axis=1)    

In [18]:
df2.shape

(51336, 54)

In [19]:
for i in df2.columns:
    df2 = df2.loc[df2[i] != -99999]

In [20]:
df2.shape

(42066, 54)

In [21]:
#checking null values
print("\nNull values in df1:")  
print(df1.isnull().sum())   
print("\nNull values in df2:")
print(df2.isnull().sum())


Null values in df1:
PROSPECTID              0
Total_TL                0
Tot_Closed_TL           0
Tot_Active_TL           0
Total_TL_opened_L6M     0
Tot_TL_closed_L6M       0
pct_tl_open_L6M         0
pct_tl_closed_L6M       0
pct_active_tl           0
pct_closed_tl           0
Total_TL_opened_L12M    0
Tot_TL_closed_L12M      0
pct_tl_open_L12M        0
pct_tl_closed_L12M      0
Tot_Missed_Pmnt         0
Auto_TL                 0
CC_TL                   0
Consumer_TL             0
Gold_TL                 0
Home_TL                 0
PL_TL                   0
Secured_TL              0
Unsecured_TL            0
Other_TL                0
Age_Oldest_TL           0
Age_Newest_TL           0
dtype: int64

Null values in df2:
PROSPECTID                    0
time_since_recent_payment     0
num_times_delinquent          0
max_recent_level_of_deliq     0
num_deliq_6mts                0
num_deliq_12mts               0
num_deliq_6_12mts             0
num_times_30p_dpd             0
num_times_60p

In [22]:
#mergigng the datasets
for i in list(df1.columns):
    if i not in df2.columns:
        print(i)

Total_TL
Tot_Closed_TL
Tot_Active_TL
Total_TL_opened_L6M
Tot_TL_closed_L6M
pct_tl_open_L6M
pct_tl_closed_L6M
pct_active_tl
pct_closed_tl
Total_TL_opened_L12M
Tot_TL_closed_L12M
pct_tl_open_L12M
pct_tl_closed_L12M
Tot_Missed_Pmnt
Auto_TL
CC_TL
Consumer_TL
Gold_TL
Home_TL
PL_TL
Secured_TL
Unsecured_TL
Other_TL
Age_Oldest_TL
Age_Newest_TL


In [23]:
df = pd.merge(df1, df2, how = 'inner', left_on=['PROSPECTID'], right_on=['PROSPECTID'])

In [24]:
print("\nShape of merged dataset:", df.shape)


Shape of merged dataset: (42066, 79)


In [31]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 42066 entries, 0 to 42065
Data columns (total 79 columns):
 #   Column                      Non-Null Count  Dtype  
---  ------                      --------------  -----  
 0   PROSPECTID                  42066 non-null  int64  
 1   Total_TL                    42066 non-null  int64  
 2   Tot_Closed_TL               42066 non-null  int64  
 3   Tot_Active_TL               42066 non-null  int64  
 4   Total_TL_opened_L6M         42066 non-null  int64  
 5   Tot_TL_closed_L6M           42066 non-null  int64  
 6   pct_tl_open_L6M             42066 non-null  float64
 7   pct_tl_closed_L6M           42066 non-null  float64
 8   pct_active_tl               42066 non-null  float64
 9   pct_closed_tl               42066 non-null  float64
 10  Total_TL_opened_L12M        42066 non-null  int64  
 11  Tot_TL_closed_L12M          42066 non-null  int64  
 12  pct_tl_open_L12M            42066 non-null  float64
 13  pct_tl_closed_L12M          420

In [25]:
df.head()

,PROSPECTID,Total_TL,Tot_Closed_TL,Tot_Active_TL,Total_TL_opened_L6M,Tot_TL_closed_L6M,pct_tl_open_L6M,pct_tl_closed_L6M,pct_active_tl,pct_closed_tl,...,pct_PL_enq_L6m_of_L12m,pct_CC_enq_L6m_of_L12m,pct_PL_enq_L6m_of_ever,pct_CC_enq_L6m_of_ever,HL_Flag,GL_Flag,last_prod_enq2,first_prod_enq2,Credit_Score,Approved_Flag
0,1,5,4,1,0,0,0.000,0.0,0.200,0.800,...,0.0,0.0,0.000,0.0,1,0,PL,PL,696,P2
1,2,1,0,1,0,0,0.000,0.0,1.000,0.000,...,0.0,0.0,0.000,0.0,0,0,ConsumerLoan,ConsumerLoan,685,P2
2,3,8,0,8,1,0,0.125,0.0,1.000,0.000,...,0.0,0.0,0.000,0.0,1,0,ConsumerLoan,others,693,P2
3,5,3,2,1,0,0,0.000,0.0,0.333,0.667,...,0.0,0.0,0.000,0.0,0,0,AL,AL,753,P1
4,6,6,5,1,0,0,0.000,0.0,0.167,0.833,...,1.0,0.0,0.429,0.0,1,0,ConsumerLoan,PL,668,P3


In [27]:
#checking for null value
df.isna().sum().sum()

0

In [28]:
#check how many columns are categorical
for i in df.columns:
    if df[i].dtype == 'object':
        print(i)

MARITALSTATUS
EDUCATION
GENDER
last_prod_enq2
first_prod_enq2
Approved_Flag


In [29]:
#values counr in categorical columns
for i in df.columns:
    if df[i].dtype == 'object':
        print(f"\nValue counts for column '{i}':")
        print(df[i].value_counts())


Value counts for column 'MARITALSTATUS':
MARITALSTATUS
Married    30887
Single     11179
Name: count, dtype: int64

Value counts for column 'EDUCATION':
EDUCATION
GRADUATE          14142
12TH              11703
SSC                7241
UNDER GRADUATE     4572
OTHERS             2291
POST-GRADUATE      1898
PROFESSIONAL        219
Name: count, dtype: int64

Value counts for column 'GENDER':
GENDER
M    37346
F     4720
Name: count, dtype: int64

Value counts for column 'last_prod_enq2':
last_prod_enq2
ConsumerLoan    16480
others          13655
PL               7553
CC               2195
AL               1353
HL                830
Name: count, dtype: int64

Value counts for column 'first_prod_enq2':
first_prod_enq2
others          20641
ConsumerLoan    11075
PL               4432
AL               2641
CC               1988
HL               1289
Name: count, dtype: int64

Value counts for column 'Approved_Flag':
Approved_Flag
P2    25454
P3     6440
P4     5264
P1     4908
Name: count, d

In [32]:
#chi square test for categorical variables
for i in ['MARITALSTATUS','EDUCATION','GENDER','last_prod_enq2','first_prod_enq2']:
    chi2,pval,_,_=chi2_contingency(pd.crosstab(df[i], df['Approved_Flag']))
    print(i,'---',pval)

MARITALSTATUS --- 3.6401547776371343e-233
EDUCATION --- 2.8651007585199264e-30
GENDER --- 1.8610038357046495e-05
last_prod_enq2 --- 0.0
first_prod_enq2 --- 8.375872889007586e-287


* since all the categorical features have pval <=0.05, we will accept all

In [33]:
#VIF for numerical variables
numerical_columns=[]
for i in df.columns:
    if df[i].dtype != 'object'and i not in ['PROSPECTID','Approved_Flag']:
        numerical_columns.append(i)

print("\nNumerical columns for VIF calculation:", numerical_columns)


Numerical columns for VIF calculation: ['Total_TL', 'Tot_Closed_TL', 'Tot_Active_TL', 'Total_TL_opened_L6M', 'Tot_TL_closed_L6M', 'pct_tl_open_L6M', 'pct_tl_closed_L6M', 'pct_active_tl', 'pct_closed_tl', 'Total_TL_opened_L12M', 'Tot_TL_closed_L12M', 'pct_tl_open_L12M', 'pct_tl_closed_L12M', 'Tot_Missed_Pmnt', 'Auto_TL', 'CC_TL', 'Consumer_TL', 'Gold_TL', 'Home_TL', 'PL_TL', 'Secured_TL', 'Unsecured_TL', 'Other_TL', 'Age_Oldest_TL', 'Age_Newest_TL', 'time_since_recent_payment', 'num_times_delinquent', 'max_recent_level_of_deliq', 'num_deliq_6mts', 'num_deliq_12mts', 'num_deliq_6_12mts', 'num_times_30p_dpd', 'num_times_60p_dpd', 'num_std', 'num_std_6mts', 'num_std_12mts', 'num_sub', 'num_sub_6mts', 'num_sub_12mts', 'num_dbt', 'num_dbt_6mts', 'num_dbt_12mts', 'num_lss', 'num_lss_6mts', 'num_lss_12mts', 'recent_level_of_deliq', 'tot_enq', 'CC_enq', 'CC_enq_L6m', 'CC_enq_L12m', 'PL_enq', 'PL_enq_L6m', 'PL_enq_L12m', 'time_since_recent_enq', 'enq_L12m', 'enq_L6m', 'enq_L3m', 'AGE', 'NETMONT

In [36]:
#VIF Sequential Check
vif_data = df[numerical_columns]
total_columns = vif_data.shape[1]
columns_to_be_kept=[]
column_index = 0

for i in range(0,total_columns):
    vif_value = variance_inflation_factor(vif_data,column_index)
    print(column_index,'---',vif_value)

    if vif_value < 6:
        columns_to_be_kept.append(numerical_columns[i])
        column_index += 1
    else:
        vif_data = vif_data.drop(numerical_columns[i], axis=1)

c:\Users\Admin\anaconda3\envs\fp\lib\site-packages\statsmodels\stats\outliers_influence.py:197: RuntimeWarning: divide by zero encountered in scalar divide
  vif = 1. / (1. - r_squared_i)


0 --- inf


c:\Users\Admin\anaconda3\envs\fp\lib\site-packages\statsmodels\stats\outliers_influence.py:197: RuntimeWarning: divide by zero encountered in scalar divide
  vif = 1. / (1. - r_squared_i)


0 --- inf
0 --- 11.319677860911183
0 --- 8.291829772566755
0 --- 6.520138234382178
0 --- 5.059265701555735
1 --- 2.6064779388021035


c:\Users\Admin\anaconda3\envs\fp\lib\site-packages\statsmodels\stats\outliers_influence.py:197: RuntimeWarning: divide by zero encountered in scalar divide
  vif = 1. / (1. - r_squared_i)


2 --- inf
2 --- 1779.2530019989363
2 --- 8.57605846135607
2 --- 3.8295656471637
3 --- 5.4692110955705875
4 --- 5.503957502932511
5 --- 1.973712154119323


c:\Users\Admin\anaconda3\envs\fp\lib\site-packages\statsmodels\stats\outliers_influence.py:197: RuntimeWarning: divide by zero encountered in scalar divide
  vif = 1. / (1. - r_squared_i)


6 --- inf
6 --- 4.811919298953729
7 --- 23.146961044025943
7 --- 30.634993848319926
7 --- 4.384922735753697
8 --- 3.064866354116124
9 --- 2.8932461784347443
10 --- 4.369654303665843
11 --- 2.208341258584075
12 --- 566.1909055769335
12 --- 1.0006593766842498
13 --- 1.9695830645742571
14 --- 7.841218783259259
14 --- 5.247703415624753


c:\Users\Admin\anaconda3\envs\fp\lib\site-packages\statsmodels\stats\outliers_influence.py:197: RuntimeWarning: divide by zero encountered in scalar divide
  vif = 1. / (1. - r_squared_i)


15 --- inf
15 --- 7.377956585088547
15 --- 1.425793366814442
16 --- 8.084822288044935
16 --- 1.622669921839381
17 --- 7.238615905249871
17 --- 15.645211505952748
17 --- 1.8184084209088587
18 --- 1.5055024528703038
19 --- 2.172012934391218
20 --- 2.6235392775473536
21 --- 2.292927625739704
22 --- 7.3588119016168045
22 --- 2.1583860086471636
23 --- 2.8651983640201
24 --- 6.457951397444763
24 --- 2.8464061660262145
25 --- 4.751462405159283
26 --- 16.664353050297613
26 --- 6.433862464128575
26 --- 8.905695279309262
26 --- 2.3948438776920553
27 --- 8.625154493499016
27 --- 13.097583499300075
27 --- 3.5102775257188883
28 --- 1.84612473749155
29 --- 18.35043652870675
29 --- 10.708249343242189
29 --- 2.3460067597985903
30 --- 21.54248963193387
30 --- 2.796240194688435
31 --- 3.3738625526342503
32 --- 9.973017265651695
32 --- 6.092683218698291
32 --- 1.0011742340538423
33 --- 3.0642351806835473
34 --- 2.807411739205449
35 --- 20.28118581063136
35 --- 15.881630139889586
35 --- 1.8328052344483385

In [38]:
print("\nColumns to be kept after VIF check:", columns_to_be_kept)
print("Number of columns to be kept:", len(columns_to_be_kept))


Columns to be kept after VIF check: ['pct_tl_open_L6M', 'pct_tl_closed_L6M', 'Tot_TL_closed_L12M', 'pct_tl_open_L12M', 'pct_tl_closed_L12M', 'Tot_Missed_Pmnt', 'CC_TL', 'Home_TL', 'PL_TL', 'Secured_TL', 'Unsecured_TL', 'Other_TL', 'Age_Newest_TL', 'time_since_recent_payment', 'max_recent_level_of_deliq', 'num_deliq_6_12mts', 'num_times_60p_dpd', 'num_std_12mts', 'num_sub', 'num_sub_6mts', 'num_sub_12mts', 'num_dbt', 'num_dbt_12mts', 'num_lss', 'num_lss_12mts', 'recent_level_of_deliq', 'CC_enq_L12m', 'PL_enq_L12m', 'time_since_recent_enq', 'enq_L3m', 'NETMONTHLYINCOME', 'Time_With_Curr_Empr', 'pct_currentBal_all_TL', 'CC_Flag', 'PL_Flag', 'pct_PL_enq_L6m_of_ever', 'pct_CC_enq_L6m_of_ever', 'HL_Flag', 'GL_Flag']
Number of columns to be kept: 39


* Initially there were 72  numerical features out of which only 39 after VIF are selected 

In [39]:
 #check anova for column_to_be_kept
from scipy.stats import f_oneway

column_to_be_kept_numerical = []

for i in columns_to_be_kept:
    a = list(df[i])
    b = list(df['Approved_Flag'])

    group_P1 = [value for value, group in zip(a, b) if group == 'P1']
    group_P2 = [value for value, group in zip(a, b) if group == 'P2']
    group_P3 = [value for value, group in zip(a, b) if group == 'P3']
    group_P4 = [value for value, group in zip(a, b) if group == 'P4']

    f_statistic, p_value = f_oneway(group_P1, group_P2, group_P3, group_P4)

    if p_value < 0.05:
        column_to_be_kept_numerical.append(i)

In [40]:
print("\nColumns to be kept after anova check:", column_to_be_kept_numerical)
print("Number of columns to be kept:", len(column_to_be_kept_numerical))


Columns to be kept after anova check: ['pct_tl_open_L6M', 'pct_tl_closed_L6M', 'Tot_TL_closed_L12M', 'pct_tl_open_L12M', 'pct_tl_closed_L12M', 'Tot_Missed_Pmnt', 'CC_TL', 'Home_TL', 'PL_TL', 'Secured_TL', 'Unsecured_TL', 'Other_TL', 'time_since_recent_payment', 'max_recent_level_of_deliq', 'num_deliq_6_12mts', 'num_times_60p_dpd', 'num_std_12mts', 'num_sub', 'num_sub_6mts', 'num_sub_12mts', 'num_dbt', 'num_dbt_12mts', 'num_lss', 'recent_level_of_deliq', 'CC_enq_L12m', 'PL_enq_L12m', 'time_since_recent_enq', 'enq_L3m', 'NETMONTHLYINCOME', 'Time_With_Curr_Empr', 'CC_Flag', 'PL_Flag', 'pct_PL_enq_L6m_of_ever', 'pct_CC_enq_L6m_of_ever', 'HL_Flag', 'GL_Flag']
Number of columns to be kept: 36
